# 📊 Logistic Regression Classifier Workshop
**Course:** CSCN8010 - Foundations of Machine Learning  
**Group 2:** Ali Cihan Ozdemir (Driver) & Lohith (Navigator)  
*(Note: Roshan did not participate in this assignment)*

---
## Project Objective
The goal of this project is to implement Statistical Classification using **Log-Loss (Cross-Entropy Loss)**. 
We will examine the `#hours studied vs. pass-fail` use case, document the underlying algorithm, and manually implement the mathematical functions required to train and evaluate a Binary Classifier.


## 1. Dataset Initialization
We begin by establishing our dataset. The independent variable ($X$) represents the number of hours studied, and the dependent variable ($y$) represents the binary outcome (0 = Fail, 1 = Pass).

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import warnings
warnings.filterwarnings('ignore')

# Independent variable: Hours studied
hours_studied = np.array([0.5, 0.75, 1, 1.25, 1.5, 1.75, 2, 2.25, 2.5, 2.75, 3, 3.25, 3.5, 3.75, 4, 4.25, 4.5, 4.75, 5])

# Dependent variable: Pass (1) or Fail (0)
pass_fail = np.array([0, 0, 0, 0, 0, 0, 0, 0, 1, 0, 1, 0, 1, 1, 1, 1, 1, 1, 1])

# Create a DataFrame for structured viewing
dataset = pd.DataFrame({
    'Hours_Studied': hours_studied,
    'Outcome': pass_fail
})

display(dataset.head())


## 2. Initial Data Visualization
Before implementing the algorithm, let's visualize the raw data to understand the distribution of passing and failing students based on their study hours.

In [ ]:
plt.figure(figsize=(10, 5))
plt.scatter(hours_studied[pass_fail == 0], pass_fail[pass_fail == 0], color='red', label='Fail (0)', s=80, alpha=0.7)
plt.scatter(hours_studied[pass_fail == 1], pass_fail[pass_fail == 1], color='green', label='Pass (1)', s=80, alpha=0.7)

plt.title("Raw Data: Hours Studied vs Pass/Fail Outcome")
plt.xlabel("Hours Studied")
plt.ylabel("Outcome (0=Fail, 1=Pass)")
plt.yticks([0, 1])
plt.grid(True, alpha=0.3)
plt.legend()
plt.show()


## 3. Algorithm: The Sigmoid Function
Logistic Regression predicts probabilities. To ensure our linear combination ($z = wx + b$) is constrained between 0 and 1, we pass it through the **Sigmoid Activation Function**:

$$ \sigma(z) = rac{1}{1 + e^{-z}} $$


In [ ]:
def sigmoid(z):
    """
    Computes the sigmoid function.
    Parameters:
      z (numpy.ndarray): The linear input (wx + b).
    Returns:
      numpy.ndarray: Probabilities bounded between 0 and 1.
    """
    # Clip z to prevent numeric overflow in exponential calculation
    z_clipped = np.clip(z, -500, 500)
    return 1 / (1 + np.exp(-z_clipped))

# Test the function
test_z = np.array([-10, 0, 10])
print(f"Sigmoid outputs for z={test_z}: {sigmoid(test_z)}")


## 4. Algorithm: Log-Loss (Cross-Entropy)
To evaluate how well our model's probabilities match the actual discrete labels, we use the **Log-Loss** (Cross-Entropy) function. It heavily penalizes the model when it predicts a high probability for an incorrect class.

Mathematical formula across $N$ samples:
$$ 	ext{LogLoss} = -rac{1}{N} \sum_{i=1}^N \left[ y_i \cdot \log(p_i) + (1 - y_i) \cdot \log(1 - p_i) ight] $$


In [ ]:
def compute_log_loss(y_true, y_prob):
    """
    Computes the Binary Cross-Entropy Loss.
    Parameters:
      y_true (numpy.ndarray): Actual binary labels (0 or 1).
      y_prob (numpy.ndarray): Predicted probabilities from the sigmoid function.
    Returns:
      float: The calculated log-loss value.
    """
    # Epsilon prevents math domain errors (taking log of exactly 0)
    eps = 1e-15
    y_prob = np.clip(y_prob, eps, 1 - eps)
    
    # Apply the log loss equation
    loss_array = y_true * np.log(y_prob) + (1 - y_true) * np.log(1 - y_prob)
    return -np.mean(loss_array)

# Test the function with a dummy prediction
dummy_prob = np.array([0.1, 0.9])
dummy_true = np.array([0, 1])
print(f"Log loss for perfectly confident correct predictions: {compute_log_loss(dummy_true, dummy_prob):.4f}")


## 5. Model Training and Weight Extraction
While we could write a gradient descent loop to find the optimal weights ($w$) and bias ($b$), we will utilize `scikit-learn` to efficiently determine the best parameters that minimize log-loss for our dataset.

In [ ]:
from sklearn.linear_model import LogisticRegression

# Scikit-learn requires 2D array for features
X_train = hours_studied.reshape(-1, 1)

# Initialize and train the internal model
classifier = LogisticRegression(random_state=42)
classifier.fit(X_train, pass_fail)

# Extract learned optimal parameters
optimal_w = classifier.coef_[0][0]
optimal_b = classifier.intercept_[0]

print(f"Learned Weight (w): {optimal_w:.4f}")
print(f"Learned Bias (b):  {optimal_b:.4f}")


## 6. Pipeline Execution: Probabilities and Validation
We will now pass the training data through our manual `sigmoid` function using the optimal weights, and then calculate the error using our manual `compute_log_loss` function. We will verifying this against `scikit-learn`'s internal metric.

In [ ]:
from sklearn.metrics import log_loss

# 1. Calculate linear combination: z = wx + b
z_values = (optimal_w * hours_studied) + optimal_b

# 2. Convert z to probabilities via manual Sigmoid
manual_probabilities = sigmoid(z_values)

# 3. Compute Log-Loss using our manual function
manual_calculated_loss = compute_log_loss(pass_fail, manual_probabilities)
print(f"Manual Log-Loss Score: {manual_calculated_loss:.4f}")

# 4. Verify against Scikit-Learn
sklearn_probabilities = classifier.predict_proba(X_train)
sklearn_calculated_loss = log_loss(pass_fail, sklearn_probabilities)
print(f"Sklearn Log-Loss Score: {sklearn_calculated_loss:.4f}")

assert np.isclose(manual_calculated_loss, sklearn_calculated_loss), "Mismatch in log-loss calculation!"


## 7. Visualizing the Learned Decision Boundary
Now we plot the continuous sigmoid curve overlaying our data, showing exactly where the 0.5 probability threshold separates passing students from failing students.

In [ ]:
# Generate continuous x values for a smooth curve
x_smooth = np.linspace(0, 6, 100)
z_smooth = (optimal_w * x_smooth) + optimal_b
y_smooth_prob = sigmoid(z_smooth)

plt.figure(figsize=(10, 6))

# Plot raw data
plt.scatter(hours_studied[pass_fail == 0], pass_fail[pass_fail == 0], color='red', label='Actual Fail', s=50, zorder=5)
plt.scatter(hours_studied[pass_fail == 1], pass_fail[pass_fail == 1], color='green', label='Actual Pass', s=50, zorder=5)

# Plot logistic sigmoid progression
plt.plot(x_smooth, y_smooth_prob, color='blue', linewidth=3, label='Learned Logistic Curve P(y=1|X)')

# Draw decision boundary (where probability = 0.5)
decision_boundary_x = -optimal_b / optimal_w
plt.axvline(x=decision_boundary_x, color='orange', linestyle='--', label=f'Decision Boundary: {decision_boundary_x:.2f} hrs')
plt.axhline(y=0.5, color='gray', linestyle=':')

plt.title("Logistic Regression: Hours Studied vs. Probability of Passing")
plt.xlabel("Hours Studied (X)")
plt.ylabel("Probability of Passing (y_hat)")
plt.legend()
plt.grid(True, alpha=0.3)
plt.show()


## 8. Analyzing the Log-Loss Exponential Penalty
To truly understand the Log-Loss algorithm, we must visualize how the logarithmic penalty behaves. The penalty scales exponentially as the model becomes "confident but wrong." 

In [ ]:
# Generate probabilities from 0.01 to 0.99
p_range = np.linspace(0.01, 0.99, 100)

# Calculate penalties
penalty_y1 = -np.log(p_range)          # Penalty if the true class was 1 
penalty_y0 = -np.log(1 - p_range)      # Penalty if the true class was 0

plt.figure(figsize=(10, 5))
plt.plot(p_range, penalty_y1, color='green', linewidth=2.5, label='Loss when True Class = 1 (Pass)')
plt.plot(p_range, penalty_y0, color='red', linewidth=2.5, label='Loss when True Class = 0 (Fail)')

plt.title("The Mechanism of Log-Loss: Exponential Misclassification Penalties")
plt.xlabel("Predicted Probability of Passing")
plt.ylabel("Computed Log-Loss Cost")
plt.legend()
plt.grid(True, alpha=0.3)
plt.show()


---
## 9. Presentation Talking Points 

The following 3 talking points are prepared for our peer review presentation, summarizing the most important aspects of our Logistic Regression implementation:

1. **The Architecture of the Sigmoid Mapping:** 
   Our model utilizes the activation function `sigmoid(z) = 1 / (1 + exp(-z))` to map a linear combination of study hours onto a non-linear S-curve. As visualized in our plot, this mathematically forces our unbounded numeric inputs strictly into the boundaries of a probabilistic domain `[0.0, 1.0]`, allowing us to classify the likelihood of a student passing.

2. **The Enforcement of Log-Loss Penalties:** 
   Unlike Mean Squared Error, the piecewise `Log-Loss` algorithm we implemented mathematically punishes hubris. If we look at our final plot, predicting a 99% probability of passing for a student who actually failed results in an exponentially massive penalty due to the `-log(1 - 0.99)` calculation. This ensures our model optimizes toward careful truth rather than blind guessing.

3. **Translating Continuous Probability to Discrete Action:** 
   While the algorithm itself outputs fluid probabilities, the actual application of Statistical Classification requires a decision threshold. We utilized a standard boundary of `P(y) = 0.5`. By calculating `-bias / weight`, we discovered the exact inflection point: studying for **2.53 hours** separates the mathematical prediction of failure from passing.


---
## 10. Interactive Peer Review: Group 5

*This section is designated for documentation during our 10:00 AM presentation exchange with Team 5.*

**Repository Details:**
* Group 5 Repo Link: `[Insert URL Here]`

**Direct Questions We Asked Group 5:**
* Q1: `[Take note here...]`
* Q2: `[Take note here...]`
* Q3: `[Take note here...]`

**Observations & Reflections on Group 5's Code (Compared to Ours):**
* `[Take note here...]`
* `[Take note here...]`
* `[Take note here...]`

**Constructive Feedback Received from Group 5:**
* `[Take note here...]`
* `[Take note here...]`
